In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
import evaluate
from peft import LoraConfig, get_peft_model, TaskType
from collections import Counter
from transformers import BitsAndBytesConfig
from collections import Counter

from dotenv import load_dotenv
import os
load_dotenv()
YOUR_HF_TOKEN = os.getenv("YOUR_HF_TOKEN")

/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def combine_text(row):
    # Collect all existing description parts in order
    desc_parts = [row.get(f"case_description_{i}", "") for i in range(1, 9) if row.get(f"case_description_{i}")]
    # Collect all existing justification parts in order
    just_parts = [row.get(f"justification_{i}", "") for i in range(1, 5) if row.get(f"justification_{i}")]
    
    # Prioritize description by putting it first
    return " ".join(desc_parts + just_parts)

In [3]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
            
        loss_fct = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # predictions = (logits > 0).astype(int)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
        "precision_macro": precision_score(labels, predictions, average="macro"),
        "recall_macro": recall_score(labels, predictions, average="macro")
    }

In [4]:
def plot_history(log_history):
    train_loss = []
    val_loss = []
    val_acc = []
    val_f1 = []
    
    for log in log_history:
        if "loss" in log and "eval_loss" not in log:
            train_loss.append(log["loss"])
    
        if "eval_loss" in log:
            val_loss.append(log["eval_loss"])
    
        if "eval_accuracy" in log:
            val_acc.append(log["eval_accuracy"])
    
        if "eval_f1_macro" in log:
            val_f1.append(log["eval_f1_macro"])
    
    
    # Convert to numpy arrays if needed
    train_loss = np.array(train_loss)
    val_loss = np.array(val_loss)
    val_acc = np.array(val_acc)
    val_f1 = np.array(val_f1)
    
    # X axes
    train_steps = np.arange(len(train_loss))
    val_steps = np.linspace(0, len(train_loss)-1, len(val_loss))
    
    # Create figure
    plt.figure(figsize=(12, 5))
    # plt.title("Supreme Court Decisions - JurBERT - 3 labels")
    
    # ----- Left subplot: Train / Val Loss -----
    plt.subplot(1, 2, 1)
    plt.plot(train_steps, train_loss, marker='o', label="Train Loss", linewidth=2)
    plt.plot(val_steps, val_loss, marker='o', label="Val Loss", linewidth=2)
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # ----- Right subplot: Val Acc / Val F1 -----
    plt.subplot(1, 2, 2)
    plt.plot(val_steps, val_acc, marker='o', label="Val Accuracy", linewidth=2)
    plt.plot(val_steps, val_f1, marker='o', label="Val F1 Macro", linewidth=2)
    plt.xlabel("Epochs")
    plt.ylabel("Score")
    plt.title("Validation Accuracy & F1")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
with open("data/regional-court-data.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

if isinstance(data, list):
    labels = [item["label"] for item in data if "label" in item]
    
    label_counts = Counter(labels)
    
    print("Number of samples per label:")
    for label, count in sorted(label_counts.items()):
        print(f"{label:12} : {count:4d} samples")
    
    print(f"\nTotal samples in Regional Court Data: {len(data)}")

Number of samples per label:
nevinovat    : 3626 samples
vinovat      : 5244 samples

Total samples in Regional Court Data: 8870


In [7]:
# Define your mapping
path = "data/regional-court-data.json"
label_map = {
    "vinovat": 0,
    "nevinovat": 1
}

def prepare_dataset(json_file_path):
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    formatted_data = []
    for entry in data:
        # Combine description (1-8) and justification (1-4)
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()
        # just = " ".join([entry.get(f"justification_{i}", "") for i in range(1, 5)]).strip()
        
        formatted_data.append({
            "text": f"{desc}", # Description prioritized by order
            "label": label_map[entry["label"]]
        })
    
    # Split into train and validation (85/15)
    train_data, val_data = train_test_split(
        formatted_data, 
        test_size=0.15, 
        stratify=[d["label"] for d in formatted_data], # Keep class ratios same
        random_state=42
    )
    
    return Dataset.from_list(train_data), Dataset.from_list(val_data)

train_raw, val_raw = prepare_dataset(path)

In [8]:
print(len(train_raw), len(val_raw))

7539 1331


## JurBERT

In [ ]:
model_name = "readerbench/jurBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        stride=256,
        return_overflowing_tokens=True,
        padding="max_length"
    )
    
    # Since one case can become 3-4 chunks, we must duplicate the label for each chunk
    sample_mapping = tokenized_inputs.pop("overflow_to_sample_mapping")
    labels = examples["label"]
    tokenized_inputs["labels"] = [labels[i] for i in sample_mapping]
    
    return tokenized_inputs

tokenized_train = train_raw.map(
    tokenize_and_align_labels, 
    batched=True, 
    remove_columns=train_raw.column_names
)

tokenized_val = val_raw.map(
    tokenize_and_align_labels, 
    batched=True, 
    remove_columns=val_raw.column_names
)

print(f"Original train cases: {len(train_raw)}")
print(f"Tokenized train windows: {len(tokenized_train)}")

The OrderedVocab you are attempting to save contains holes for indices [6, 7, 10, 41], your vocabulary could be corrupted!


Map: 100%|██████████| 7539/7539 [00:10<00:00, 749.72 examples/s]


The OrderedVocab you are attempting to save contains holes for indices [6, 7, 10, 41], your vocabulary could be corrupted!


Map: 100%|██████████| 1331/1331 [00:01<00:00, 702.00 examples/s]

Original train cases: 7539
Tokenized train windows: 7539


In [ ]:
# # compute label counts in train and validation sets and print with original label names
# inv_label_map = {v: k for k, v in label_map.items()}

# train_counts = Counter(train_raw["label"])
# val_counts = Counter(val_raw["label"])

# print(f"Train total: {len(train_raw)}")
# for label_id, cnt in sorted(train_counts.items()):
#     print(f"{inv_label_map.get(label_id, label_id):12} : {cnt}")

# print(f"\nVal total:   {len(val_raw)}")
# for label_id, cnt in sorted(val_counts.items()):
#     print(f"{inv_label_map.get(label_id, label_id):12} : {cnt}")

Train total: 7539
vinovat      : 4457
nevinovat    : 3082

Val total:   1331
vinovat      : 787
nevinovat    : 544


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
    ).to(device) 
print(f"Using device: {device}")

training_args = TrainingArguments(
    output_dir="results/regional-jurbert",
    learning_rate=2e-6,
    lr_scheduler_type="reduce_lr_on_plateau",
    warmup_steps=500,
    warmup_ratio=0.1,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.1,
    fp16=True,
    logging_steps=500,
    eval_steps=500,
    eval_strategy="steps",
    save_strategy="epoch",
    metric_for_best_model="f1_macro", 
    greater_is_better=True,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train, # Your processed dataset
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
# trainer.train()

# log_history = trainer.state.log_history
# plot_history(log_history)

## RoBERT

In [ ]:
model_name = "readerbench/RoBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        stride=256,
        return_overflowing_tokens=True,
        padding="max_length"
    )
    
    # Since one case can become 3-4 chunks, we must duplicate the label for each chunk
    sample_mapping = tokenized_inputs.pop("overflow_to_sample_mapping")
    labels = examples["label"]
    tokenized_inputs["labels"] = [labels[i] for i in sample_mapping]
    
    return tokenized_inputs

tokenized_train = train_raw.map(
    tokenize_and_align_labels, 
    batched=True, 
    remove_columns=train_raw.column_names
)

tokenized_val = val_raw.map(
    tokenize_and_align_labels, 
    batched=True, 
    remove_columns=val_raw.column_names
)

print(f"Original train cases: {len(train_raw)}")
print(f"Tokenized train windows: {len(tokenized_train)}")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
    ).to(device) 
print(f"Using device: {device}")

training_args = TrainingArguments(
    output_dir="results/regional-robert",
    learning_rate=1e-5,
    lr_scheduler_type="reduce_lr_on_plateau",
    warmup_steps=300,
    warmup_ratio=0.1,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.1,
    fp16=True,
    logging_steps=500,
    eval_steps=500,
    eval_strategy="steps",
    save_strategy="epoch",
    metric_for_best_model="f1_macro", 
    greater_is_better=True,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train, # Your processed dataset
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
# trainer.train()

# log_history = trainer.state.log_history
# plot_history(log_history)

## BERTBase

In [ ]:
model_name = "dumitrescustefan/bert-base-romanian-cased-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # tokenizer.pad_token = tokenizer.eos_token
    # tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 1331/1331 [00:01<00:00, 875.41 examples/s]


In [ ]:
# def token_stats(dataset):
#     if "attention_mask" in dataset.column_names:
#         lengths = [int(sum(mask)) for mask in dataset["attention_mask"]]
#     else:
#         pad_id = tokenizer.pad_token_id
#         lengths = [int(np.sum(np.array(ids) != pad_id)) for ids in dataset["input_ids"]]
    
#     return {
#         "avg_tokens": float(np.mean(lengths)),
#         "max_tokens": int(np.max(lengths)),
#         "min_tokens": int(np.min(lengths)),
#     }

# train_stats = token_stats(tokenized_train)
# val_stats = token_stats(tokenized_val)

# print("tokenized_train stats:", train_stats)
# print("tokenized_val stats:", val_stats)

tokenized_train stats: {'avg_tokens': 3328.091524074811, 'max_tokens': 27875, 'min_tokens': 23}
tokenized_val stats: {'avg_tokens': 3407.6453794139743, 'max_tokens': 24433, 'min_tokens': 18}


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
    ).to(device) 
print(f"Using device: {device}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dumitrescustefan/bert-base-romanian-cased-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cpu


In [ ]:
# training_args = TrainingArguments(
#     output_dir="results/regional-bert-llm",
#     learning_rate=2e-5,
#     lr_scheduler_type="cosine",
#     warmup_steps=200,
#     warmup_ratio=0.1,
#     per_device_train_batch_size=8,
#     gradient_accumulation_steps=2,
#     num_train_epochs=5,
#     weight_decay=0.1,
#     fp16=True,
#     logging_steps=50,
#     eval_steps=50,
#     eval_strategy="steps",
#     save_strategy="epoch",
#     report_to="none",
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_train,
#     eval_dataset=tokenized_val,
#     compute_metrics=compute_metrics,
#     data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
# )

# trainer.train()

/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
50,0.992900,0.764132,0.757143,0.722256,0.740047,0.721618
100,0.666100,0.509788,0.847619,0.823640,0.855924,0.824726
150,0.535600,0.420005,0.861905,0.835366,0.879099,0.837234
200,0.439300,0.300985,0.919048,0.911013,0.924414,0.905390
250,0.422500,0.364245,0.861905,0.849512,0.857273,0.847124
300,0.312000,0.311775,0.909524,0.899926,0.915042,0.895593
350,0.264000,0.301836,0.909524,0.901926,0.909491,0.897028


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory wo

TrainOutput(global_step=375, training_loss=0.49969359842936195, metrics={'train_runtime': 4441.4696, 'train_samples_per_second': 1.34, 'train_steps_per_second': 0.084, 'total_flos': 1565524835481600.0, 'train_loss': 0.49969359842936195, 'epoch': 5.0})

## LegalBERT

In [ ]:
model_name = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # tokenizer.pad_token = tokenizer.eos_token
    # tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 1331/1331 [00:01<00:00, 908.09 examples/s]


In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
    ).to(device) 
print(f"Using device: {device}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


In [ ]:
training_args = TrainingArguments(
    output_dir="results/good/dist-legal-bert",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.1,
    fp16=True,
    logging_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
50,0.672000,0.671828,0.595041,0.406640,0.581697,0.509136
100,0.687500,0.667065,0.601052,0.421161,0.621989,0.516489
150,0.687900,0.666768,0.600301,0.420774,0.614828,0.515854
200,0.667900,0.651127,0.619083,0.572824,0.597787,0.578279
250,0.633900,0.622041,0.664162,0.596214,0.678604,0.609588
300,0.656600,0.627365,0.649136,0.648958,0.664802,0.667262
350,0.585400,0.579561,0.706236,0.692836,0.695585,0.691140
400,0.581200,0.602347,0.679940,0.676513,0.677910,0.683945
450,0.606200,0.640021,0.663411,0.560943,0.738065,0.595046
500,0.560800,0.572341,0.718257,0.685082,0.724547,0.681724


TrainOutput(global_step=2360, training_loss=0.4506611065339234, metrics={'train_runtime': 763.9782, 'train_samples_per_second': 49.34, 'train_steps_per_second': 3.089, 'total_flos': 9917971231795200.0, 'train_loss': 0.4506611065339234, 'epoch': 5.0})

# LLMs

In [ ]:
try:
    from huggingface_hub import login
    login(YOUR_HF_TOKEN)
except Exception as e:
    print("Hugging Face login skipped:", e)

In [9]:
def prepare_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    label_map = {"respins": 0, "admis": 1}
    data = [item for item in data if item['label'] != 'inadmisibil']

    formatted_data = []
    for entry in data:
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()
        
        # --- PROMPT ENGINEERING START ---
        
        just = " ".join([entry.get(f"justification_{i}", "") for i in range(1, 5)]).strip()
        instructional_prompt = (
            f"Decide verdictul acestui caz juridic:\n\n"
            f"DESCRIERE: {desc}\n"
            f"JUSTIFICARE: {just}\n\n"
            f"Ai de ales între două răspunsuri: admis sau respins.\n"
        )

        # --- PROMPT ENGINEERING END ---
        
        formatted_data.append({
            "text": instructional_prompt, 
            "label": label_map[entry["label"]]
        })
    
    train_data, val_data = train_test_split(
        formatted_data, 
        test_size=0.15, 
        stratify=[d["label"] for d in formatted_data], # Keep class ratios same
        random_state=42
    )
    
    return Dataset.from_list(train_data), Dataset.from_list(val_data)

In [10]:
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    results = clf_metrics.compute(predictions=predictions, references=labels)
    
    decoded_preds = [str(p) for p in predictions]
    decoded_labels = [str(l) for l in labels]
    
    results.update(rouge.compute(predictions=decoded_preds, references=decoded_labels))    
    return results

In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

## Llama

In [ ]:
model_name = "meta-llama/Llama-2-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


In [ ]:
device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

In [ ]:
# training_args = TrainingArguments(
#     output_dir="results/supreme-llama-llm",
#     lr_scheduler_type="reduce_lr_on_plateau",
#     learning_rate=5e-5,
#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=4,
#     num_train_epochs=3,
#     weight_decay=0.2,
#     gradient_checkpointing=True,       # Crucial for saving VRAM
#     optim="paged_adamw_32bit",         # Memory-efficient optimizer
#     logging_steps=150,
#     eval_steps=150,
#     eval_strategy="steps",
#     save_strategy="epoch",
#     report_to="none",
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_train,
#     eval_dataset=tokenized_val,
#     compute_metrics=compute_metrics,
#     data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
# )

# trainer.train()

## RoLlama

In [12]:
model_name = "OpenLLM-Ro/RoLlama2-7b-Base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)

Map: 100%|██████████| 1331/1331 [00:00<00:00, 1592.32 examples/s]


In [13]:
from peft import prepare_model_for_kbit_training

device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float16, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
# model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

Loading checkpoint shards: 100%|██████████| 3/3 [00:04<00:00,  1.42s/it]
Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at OpenLLM-Ro/RoLlama2-7b-Base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    num_train_epochs=3,
    weight_decay=0.2,
    fp16 = True,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
)

trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum
150,2.025900,1.052582,0.513148,0.401109,0.403346,0.398897,0.513148,0.000000,0.513148,0.512397
300,1.966300,1.087472,0.552968,0.315305,0.421538,0.251838,0.552968,0.000000,0.552216,0.552968
450,2.181000,1.027227,0.571751,0.246032,0.438679,0.170956,0.571751,0.000000,0.570999,0.571751
600,1.998500,0.915047,0.564237,0.410569,0.459091,0.371324,0.564237,0.000000,0.563486,0.564237
750,2.005000,0.917499,0.600301,0.369668,0.520000,0.286765,0.600301,0.000000,0.599549,0.600301
900,1.820700,0.885188,0.607062,0.288435,0.554974,0.194853,0.606311,0.000000,0.605560,0.606311
1050,1.800900,0.828969,0.626597,0.393162,0.585455,0.295956,0.626221,0.000000,0.625845,0.627348
1200,1.680400,0.746692,0.646882,0.575812,0.565603,0.586397,0.647633,0.000000,0.646882,0.646882
1350,1.689200,0.863862,0.698723,0.546893,0.709677,0.444853,0.698723,0.000000,0.698723,0.697971
1500,2.248400,1.032438,0.666416,0.406417,0.745098,0.279412,0.665665,0.000000,0.665665,0.666416


KeyboardInterrupt: 

## RoGemma

In [12]:
model_name = "OpenLLM-Ro/RoGemma-7b-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 1331/1331 [00:00<00:00, 1457.18 examples/s]


In [13]:
from peft import prepare_model_for_kbit_training

device_map = "auto"

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float16, # Ensure model weights are in float32 for LoRA
    device_map=device_map,
    max_memory={0: "14GiB", "cpu": "48GiB"},
)
model.config.pad_token_id = tokenizer.pad_token_id
model.gradient_checkpointing_enable()
model.config.use_cache = False

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=4, 
    lora_alpha=16, 
    lora_dropout=0.1,
    target_modules=["q_proj"] # Target the attention layers
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

Loading checkpoint shards:  75%|███████▌  | 3/4 [00:05<00:01,  1.88s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 36.00 MiB. GPU 0 has a total capacity of 15.47 GiB of which 32.25 MiB is free. Process 191012 has 9.82 GiB memory in use. Including non-PyTorch memory, this process has 5.60 GiB memory in use. Of the allocated memory 5.29 GiB is allocated by PyTorch, and 87.62 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    num_train_epochs=3,
    weight_decay=0.2,
    fp16 = True,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
)

trainer.train()

/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum
150,2.099800,1.772569,0.565574,0.485437,0.568182,0.423729,0.565574,0.000000,0.565574,0.565574
300,1.999500,1.535476,0.540984,0.548387,0.523077,0.576271,0.540984,0.000000,0.540984,0.540984
450,1.822700,1.913286,0.573770,0.380952,0.640000,0.271186,0.573770,0.000000,0.573770,0.573770
600,1.827800,1.535027,0.581967,0.474227,0.605263,0.389831,0.581967,0.000000,0.581967,0.581967
750,1.614600,2.083428,0.549180,0.303797,0.600000,0.203390,0.549180,0.000000,0.549180,0.549180
900,1.779100,1.981803,0.516393,0.646707,0.500000,0.915254,0.516393,0.000000,0.516393,0.516393
1050,1.406000,1.255050,0.663934,0.655462,0.650000,0.661017,0.663934,0.000000,0.663934,0.663934
1200,1.149100,1.231574,0.655738,0.637931,0.649123,0.627119,0.655738,0.000000,0.655738,0.655738
1350,1.106800,1.240295,0.647541,0.638655,0.633333,0.644068,0.647541,0.000000,0.647541,0.647541
1500,1.253000,1.538366,0.639344,0.531915,0.714286,0.423729,0.639344,0.000000,0.639344,0.639344


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning:

TrainOutput(global_step=2070, training_loss=1.5031612562096637, metrics={'train_runtime': 1545.2975, 'train_samples_per_second': 1.34, 'train_steps_per_second': 1.34, 'total_flos': 4.931100047572992e+16, 'train_loss': 1.5031612562096637, 'epoch': 3.0})

## RoMistral

In [12]:
model_name = "OpenLLM-Ro/RoMistral-7b-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 1331/1331 [00:00<00:00, 1516.07 examples/s]


In [13]:
from peft import prepare_model_for_kbit_training

device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float16, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
# model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

Loading checkpoint shards: 100%|██████████| 3/3 [00:12<00:00,  4.10s/it]
Some weights of MistralForSequenceClassification were not initialized from the model checkpoint at OpenLLM-Ro/RoMistral-7b-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    num_train_epochs=3,
    weight_decay=0.2,
    fp16 = True,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
)

trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum
150,6.898400,3.290342,0.557476,0.221929,0.394366,0.154412,0.556724,0.000000,0.558227,0.558227
300,5.287200,4.857966,0.592787,0.055749,0.533333,0.029412,0.593539,0.000000,0.593539,0.592787
450,5.413800,2.293272,0.590533,0.206696,0.496503,0.130515,0.589782,0.000000,0.591285,0.590533
600,4.206700,1.823481,0.546957,0.443213,0.445269,0.441176,0.546582,0.000000,0.546957,0.546957
750,3.420300,2.039894,0.591285,0.000000,0.000000,0.000000,0.591285,0.000000,0.592036,0.591285
900,2.877700,0.807179,0.591285,0.000000,0.000000,0.000000,0.591285,0.000000,0.592036,0.591285
1050,2.227300,0.957001,0.522915,0.360524,0.398664,0.329044,0.522915,0.000000,0.523666,0.522915


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


KeyboardInterrupt: 

## RoGPT2

In [12]:
model_name = "readerbench/RoGPT2-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 1331/1331 [00:00<00:00, 1368.54 examples/s]


In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Important for GPT2 padding
model.config.pad_token_id = tokenizer.pad_token_id

model = model.to(device)

print(f"Using device: {device}")

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at readerbench/RoGPT2-base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


In [ ]:
training_args = TrainingArguments(
    output_dir="results/supreme-bert-llm",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    weight_decay=0.1,
    fp16=True,
    logging_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum
50,0.775000,0.744405,0.478588,0.489706,0.408088,0.612132,0.479339,0.000000,0.479339,0.478588
100,0.697200,0.700832,0.560481,0.233290,0.406393,0.163603,0.560481,0.000000,0.561983,0.561608
150,0.704500,0.699253,0.550714,0.461261,0.452297,0.470588,0.550714,0.000000,0.550714,0.551089
200,0.685900,0.678860,0.604808,0.246418,0.558442,0.158088,0.604808,0.000000,0.605560,0.605560
250,0.684800,0.656539,0.612322,0.250000,0.597222,0.158088,0.611570,0.000000,0.612322,0.612322
300,0.665700,0.640925,0.631104,0.270431,0.705426,0.167279,0.630353,0.000000,0.631856,0.631856
350,0.618400,0.586239,0.670173,0.546019,0.624113,0.485294,0.670173,0.000000,0.669421,0.670924
400,0.590800,0.592745,0.701728,0.548350,0.719403,0.443015,0.700977,0.000000,0.701728,0.702479
450,0.535700,0.591105,0.713749,0.666667,0.636060,0.700368,0.714500,0.000000,0.712246,0.713749
500,0.564400,0.550625,0.724267,0.670852,0.654991,0.687500,0.725019,0.000000,0.723516,0.724267


KeyboardInterrupt: 